In [ ]:
from datasets import load_dataset
data = load_dataset("rotten_tomatoes")
data["train"][0,-1]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

In [ ]:
data["train"][0,-1]

NameError: name 'data' is not defined

In [ ]:
from transformers import pipeline

# 我们的Hugging Face模型路径
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 将模型加载到流水线中
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [1]:
import torch
from datasets import load_dataset
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from sklearn.metrics import classification_report
from tqdm import tqdm

# 1. 加载数据集
data = load_dataset("rotten_tomatoes")

# 2. 初始化 Pipeline
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
pipe = pipeline(
    task="sentiment-analysis",
    model=model_path,
    tokenizer=model_path,
    # --- 修改点 1: 使用 top_k=None 替代 return_all_scores=True ---
    # 这确保输出是一个包含所有标签分数的列表，例如:
    # [{'label': 'negative', 'score': 0.1}, {'label': 'neutral', 'score': 0.2}, {'label': 'positive', 'score': 0.7}]
    top_k=None,
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)

# 3. 运行推理
y_pred = []

# --- 优化点: 添加 batch_size 可以显著提高 GPU 推理速度 ---
for output in tqdm(pipe(KeyDataset(data["test"], "text"), batch_size=8), total=len(data["test"])):

    # --- 修改点 2: 直接使用 output ---
    # 此时 output 已经是列表结构，不需要再取 [0]
    score_list = output

    # 获取特定标签的分数
    # 注意：该模型有三个标签：negative, neutral, positive
    negative_score = next(item["score"] for item in score_list if item["label"] == "negative")
    positive_score = next(item["score"] for item in score_list if item["label"] == "positive")

    # 简单的逻辑：比较正面和负面（忽略中性）
    # Rotten Tomatoes 是二分类：0=Negative, 1=Positive
    assignment = 0 if negative_score > positive_score else 1
    y_pred.append(assignment)

# 4. 评估
print(classification_report(
    data["test"]["label"],
    y_pred,
    target_names=["Negative", "Positive"]
))
# 从 sentence_transformers 库导入 SentenceTransformer 类
from sentence_transformers import SentenceTransformer

# 加载预训练的嵌入模型
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 将训练集文本转换为嵌入向量，并显示进度条
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)

# 将测试集文本转换为嵌入向量，并显示进度条
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
# 从 scikit-learn 导入逻辑回归模型
from sklearn.linear_model import LogisticRegression

# 基于训练嵌入向量构建逻辑回归模型
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

# 预测未见过的样本
y_pred = clf.predict(test_embeddings)

# 评估模型性能
print("Logistic Regression Performance:")
print(classification_report(
    data["test"]["label"],
    y_pred,
    target_names=["Negative", "Positive"]
))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Logistic Regression Performance:
              precision    recall  f1-score   support

    Negative       0.85      0.86      0.85       533
    Positive       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



In [2]:
label_embeddings=model.encode(["A negative review","A positive review"])

In [1]:
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm import tqdm
from sklearn.metrics import classification_report

# 1. 初始化 Pipeline
# 移除 task="text2text-generation" 参数
pipe = pipeline(
    task="text-generation",  # 您的错误信息显示列表中支持这个任务
    model="google/flan-t5-small",
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)
prompt = "Is the following sentence positive or negative? "

# 假设 data 已经在之前的步骤加载了
# 如果 data 丢失了，需要取消下面这行的注释:
from datasets import load_dataset
data = load_dataset("rotten_tomatoes")

# 2. 处理数据
# 注意：这里最好给新列起个名字，不覆盖原有的 text，虽然您的写法也可以运行
data_processed = data.map(lambda example: {"t5_input": prompt + example['text']})

# 3. 运行推理
y_pred = []

# 注意：KeyDataset 需要指向我们刚才构建的带 prompt 的列 "t5_input"
# 如果您上面用的是您的原始写法 {"t5": ...}，这里就用 "t5"
for output in tqdm(pipe(KeyDataset(data_processed["test"], "t5_input"), batch_size=8), total=len(data["test"])):
    text = output[0]["generated_text"]
    # T5 生成的文本可能包含空格或大小写差异，建议 strip() 并 lower()
    clean_text = text.strip().lower()
    y_pred.append(0 if clean_text == "negative" else 1)

# 4. 评估 (修复了函数名和参数传递方式)
print(classification_report(
    data["test"]["label"],
    y_pred,
    target_names=["Negative", "Positive"]
))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

100%|██████████| 1066/1066 [05:29<00:00,  3.23it/s]

              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00       533
    Positive       0.50      1.00      0.67       533

    accuracy                           0.50      1066
   macro avg       0.25      0.50      0.33      1066
weighted avg       0.25      0.50      0.33      1066




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
import openai

# 1. 创建OpenAI客户端
client = openai.OpenAI(api_key="A")  # 替换为你的API密钥

def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    """
    基于提示词和输入文档生成模型输出
    :param prompt: 提示词模板，包含[DOCUMENT]占位符
    :param document: 待处理的文本（如电影评论）
    :param model: 使用的模型名称
    :return: 模型生成的结果
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant."  # 系统角色设定
        },
        {
            "role": "user",
            "content": prompt.replace("[DOCUMENT]", document)  # 替换占位符为实际文本
        }
    ]

    # 调用模型生成结果
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        temperature=0  # 温度设为0，保证结果确定性，适合分类任务
    )

    return chat_completion.choices[0].message.content

# 2. 定义情感分类提示词模板
prompt = """Predict whether the following document is a positive or negative movie review:
[DOCUMENT]
If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""
# 待分类文本
document = "unpretentious, charming, quirky, original"
# 调用函数进行分类
chatgpt_generation(prompt, document)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}